# Nestle Executive HR Intelligence System v5 (Python 3.12 Safe)
### Fixes the exact ASGI crash: `TypeError: argument of type 'bool' is not iterable`
---
**v5 fix:** Monkey-patches `gradio_client.utils._json_schema_to_python_type` + `show_api=False`
to neutralise the Python 3.12 Pydantic v2 boolean schema crash in `gradio/routes.py:api_info`.

**Run order:** Cell 1 -> restart runtime -> Cell 2 -> Cell 3

In [5]:
# ── CELL 1: DEPENDENCIES ──────────────────────────────────────────────────────
!pip uninstall -y gradio huggingface_hub
%pip install --quiet --upgrade \
    huggingface_hub==0.23.4 \
    gradio==4.36.1 \
    httpx==0.27.2 \
    openai==1.51.0 \
    langchain==0.2.16 \
    langchain-openai==0.1.23 \
    langchain-community==0.2.16 \
    langchain-chroma==0.1.4 \
    chromadb==0.5.20 \
    pypdf==4.3.1 \
    tiktoken==0.7.0

print('✅ Dependencies installed.')
print('⚠️  RESTART RUNTIME NOW — Runtime → Restart session — then run Cell 2.')

Found existing installation: gradio 4.36.1
Uninstalling gradio-4.36.1:
  Successfully uninstalled gradio-4.36.1
Found existing installation: huggingface-hub 0.23.4
Uninstalling huggingface-hub-0.23.4:
  Successfully uninstalled huggingface-hub-0.23.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.23.4 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.23.4 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.23.4 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.23.4 which is incompatible.
✅ Dependencies installed.
⚠️  RESTART RUNTIME NOW — Runtime → Restart session — then run Cell 2.


In [1]:
# ── CELL 2: AUTHENTICATION + VECTOR DB BUILD ──────────────────────────────────
import os, shutil, warnings
from getpass import getpass
from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import gradio as gr

warnings.filterwarnings('ignore')

# ── Constants ─────────────────────────────────────────────────────────────────
LLM_MODEL          = 'gpt-3.5-turbo'
EMBEDDING_MODEL    = 'text-embedding-ada-002'
TOKEN_CHUNK_SIZE   = 500
TOKEN_OVERLAP      = 50
RETRIEVAL_K        = 4
TEMPERATURE        = 0.0
MAX_HISTORY_TURNS  = 5
CHROMA_PERSIST_DIR = './chroma_nestle_v4'
DOCUMENT_PATH      = 'the_nestle_hr_policy_pdf_2012.pdf'  # ← set to your uploaded PDF filename

# ── Sentinel globals — Cell 3 checks these before serving any request ─────────
# FIX 4: Declare both as None here so handle_query never hits a NameError
#         even if this cell is re-run or Cell 2 is skipped.
vector_db     = None
openai_client = None

# ── Auth ──────────────────────────────────────────────────────────────────────
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('🔐 OpenAI API Key: ').strip()

openai_client = OpenAI()   # reused on every query — no per-request construction

# ── PDF → Chunks → Chroma ─────────────────────────────────────────────────────
if not os.path.exists(DOCUMENT_PATH):
    raise FileNotFoundError(
        f"❌ '{DOCUMENT_PATH}' not found. "
        "Upload it via the Colab file panel (left sidebar)."
    )

print('📥 Loading PDF...')
pages = PyPDFLoader(DOCUMENT_PATH).load()

print('✂️  Chunking into 500-token segments...')
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name=LLM_MODEL,
    chunk_size=TOKEN_CHUNK_SIZE,
    chunk_overlap=TOKEN_OVERLAP
)
chunks = splitter.split_documents(pages)

print(f'🧠 Embedding {len(chunks)} chunks — this takes ~1 min for a large PDF...')
if os.path.exists(CHROMA_PERSIST_DIR):
    shutil.rmtree(CHROMA_PERSIST_DIR)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(model=EMBEDDING_MODEL),
    persist_directory=CHROMA_PERSIST_DIR
)

print(f'✅ Vector DB ready — {len(chunks)} chunks across {len(pages)} pages.')
print('   Run Cell 3 to launch the UI.')

🔐 OpenAI API Key: ··········
📥 Loading PDF...
✂️  Chunking into 500-token segments...
🧠 Embedding 10 chunks — this takes ~1 min for a large PDF...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector DB ready — 10 chunks across 8 pages.
   Run Cell 3 to launch the UI.


In [2]:
# ── CELL 3: GRADIO UI — PYTHON 3.12 SAFE ─────────────────────────────────────
#
# NEW in v5: Two-layer fix for the ASGI crash seen in the traceback:
#
#   File "gradio/routes.py", line 427, in api_info
#     app.api_info = app.get_blocks().get_api_info()
#   File "gradio_client/utils.py", line 863, in get_type
#     if "const" in schema:
#   TypeError: argument of type 'bool' is not iterable
#
# ROOT CAUSE: Pydantic v2 on Python 3.12 emits literal True/False as JSON
# schema nodes. Gradio 4.36.1 does not guard against non-dict schemas in
# get_type(), crashing whenever the /info API route is hit (which happens
# automatically on every page load).
#
# LAYER 1 — Monkey-patch: fix the bad line inside gradio_client at runtime.
# LAYER 2 — show_api=False: disable the /info route entirely so the patched
#            code path is never reached even if layer 1 has edge cases.

import gradio_client.utils as _gcu

_orig_schema_fn = _gcu._json_schema_to_python_type

def _safe_schema_fn(schema, defs=None):
    # Guard: if schema is bool/None/int (not a dict), return generic "any"
    # This is the exact line that crashes: `if "const" in schema` when schema=True
    if not isinstance(schema, dict):
        return "any"
    return _orig_schema_fn(schema, defs)

_gcu._json_schema_to_python_type = _safe_schema_fn
print("Compatibility patch applied — boolean schema nodes now handled safely.")

# ── System prompt ──────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are the Nestle Executive HR Intelligence Assistant.\n"
    "Answer EXCLUSIVELY from the policy context provided below.\n"
    "If the answer is absent, say: This parameter is not defined in the current "
    "policy document. Please consult HR Legal.\n"
    "Be concise, professional, and cite page number(s) at the end.\n\n"
    "Policy Context:\n{context}"
)


def retrieve_context(query):
    docs    = vector_db.similarity_search(query, k=RETRIEVAL_K)
    context = "\n\n---\n\n".join(d.page_content for d in docs)
    pages   = sorted(set(
        d.metadata["page"] + 1
        for d in docs
        if isinstance(d.metadata.get("page"), int)
    ))
    return context, pages


def build_messages(query, context, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT.format(context=context)}]
    for pair in history[-MAX_HISTORY_TURNS:]:
        if pair[0]:
            messages.append({"role": "user",      "content": pair[0]})
        if pair[1]:
            messages.append({"role": "assistant", "content": pair[1]})
    messages.append({"role": "user", "content": query})
    return messages


def handle_query(query, history):
    if vector_db is None or openai_client is None:
        msg = "System not ready. Please run Cell 2 first to build the vector DB."
        updated = history + [[query, msg]]
        return updated, updated, ""

    if not query or not query.strip():
        return history, history, ""

    try:
        context, source_pages = retrieve_context(query)
    except Exception as exc:
        msg = "Retrieval error: " + str(exc)
        updated = history + [[query, msg]]
        return updated, updated, ""

    try:
        messages = build_messages(query, context, history)
        response = openai_client.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            temperature=TEMPERATURE,
            max_tokens=1024,
        )
        answer = response.choices[0].message.content.strip()
    except Exception as exc:
        answer = "OpenAI API error: " + str(exc)

    if source_pages:
        answer += "\n\n---\n*Source: Page(s) " + ", ".join(str(p) for p in source_pages) + "*"

    updated = history + [[query, answer]]
    return updated, updated, ""


def clear_session():
    return [], [], ""


# ── Layout ────────────────────────────────────────────────────────────────────
CSS = """
.gradio-container {
    background-color: #f8f9fa !important;
    font-family: 'Helvetica Neue', Arial, sans-serif;
}
.exec-header {
    background: linear-gradient(135deg, #002f6c 0%, #0046a0 100%);
    color: white;
    padding: 28px 30px;
    border-radius: 10px;
    text-align: center;
    box-shadow: 0 4px 15px rgba(0,0,0,0.15);
    margin-bottom: 20px;
}
.exec-header h1 { margin:0; font-size:26px; font-weight:300; letter-spacing:1px; }
.exec-header p  { margin:8px 0 0; opacity:0.85; font-size:13px;
                   text-transform:uppercase; letter-spacing:1.5px; }
"""

with gr.Blocks(css=CSS, theme=gr.themes.Default()) as demo:

    gr.HTML(
        "<div class=\'exec-header\'>"
        "<h1>Nestle HR Intelligence Layer</h1>"
        "<p>C-Suite Knowledge Retrieval Engine v5</p>"
        "</div>"
    )

    chat_state  = gr.State([])
    chat_window = gr.Chatbot(height=520, show_label=False, bubble_full_width=False)

    with gr.Row():
        query_input = gr.Textbox(
            placeholder="Enter your policy inquiry here...",
            show_label=False,
            scale=5
        )
        submit_btn = gr.Button("Execute Query", variant="primary", scale=1)

    clear_btn = gr.Button("Purge Session Memory", variant="secondary")

    submit_btn.click(
        fn=handle_query,
        inputs=[query_input, chat_state],
        outputs=[chat_state, chat_window, query_input],
        concurrency_limit=1
    )
    query_input.submit(
        fn=handle_query,
        inputs=[query_input, chat_state],
        outputs=[chat_state, chat_window, query_input],
        concurrency_limit=1
    )
    clear_btn.click(
        fn=clear_session,
        inputs=[],
        outputs=[chat_state, chat_window, query_input]
    )

print("\n=======================================================")
print("LAUNCHING - open the public gradio.live URL below")
print("Do NOT use the Colab inline preview.")
print("=======================================================\n")

# show_api=False disables the /info route that contained the crashing code
demo.queue().launch(share=True, show_api=False, max_threads=4)


Compatibility patch applied — boolean schema nodes now handled safely.

LAUNCHING - open the public gradio.live URL below
Do NOT use the Colab inline preview.

IMPORTANT: You are using gradio version 4.36.1, however version 4.44.1 is available, please upgrade.
--------
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://216bddcb6d58cc1f9f.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
